# Hardware Trojan pipeline — CSAW AI Hardware Attack Challenge 2026

Run top to bottom. Every model call is logged to `ai/logs/` so the trace shows the AI wrote the hardware.

- **Setup** — install + API key (key is never written to disk)
- **Logged client** — `ask()` / `chat()` with full logging
- **Smoke test** — should print `pipeline OK`
- **Golden model** — recovered cipher, used to score generated RTL


## 1. Setup

In [ ]:
!pip -q install openai
import os, re, json, time, datetime, pathlib
from getpass import getpass

# endpoint + model (edit BASE_URL to your provider)
os.environ["LLM_MODEL"]    = "nvidia/nemotron-3-super-120b-a12b"
os.environ["LLM_BASE_URL"] = "https://integrate.api.nvidia.com/v1"   # or Portkey: https://api.portkey.ai/v1

# paste your key when prompted (not stored on disk, not committed)
if not os.environ.get("LLM_API_KEY"):
    os.environ["LLM_API_KEY"] = getpass("LLM_API_KEY: ")

# if your provider needs extra headers (e.g. Portkey virtual key), set them here:
# os.environ["LLM_EXTRA_HEADERS"] = json.dumps({"x-portkey-virtual-key": "..."})


## 2. Logged LLM client

In [ ]:
"""Logged client: every call -> ai/logs/NNN_timestamp_label.md + ai/logs/transcript.md."""
from openai import OpenAI

LOG_DIR = pathlib.Path("ai/logs"); LOG_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT = LOG_DIR / "transcript.md"
MODEL = os.environ.get("LLM_MODEL", "nvidia/nemotron-3-super-120b-a12b")
BASE  = os.environ.get("LLM_BASE_URL") or None
KEY   = os.environ["LLM_API_KEY"]
try:    HEADERS = json.loads(os.environ.get("LLM_EXTRA_HEADERS","") or "{}")
except json.JSONDecodeError: HEADERS = {}

_client = OpenAI(api_key=KEY, base_url=BASE, default_headers=HEADERS or None)
_n = 0

def _log(label, messages, reply, model, dt):
    global _n; _n += 1
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fn = LOG_DIR / f"{_n:03d}_{stamp}_{label}.md"
    with open(fn, "w") as f:
        f.write(f"# Call {_n:03d} - {label}\n\n- model: {model}\n- latency: {dt:.1f}s\n\n")
        for m in messages: f.write(f"## {m['role']}\n\n{m['content']}\n\n")
        f.write(f"## assistant (response)\n\n{reply}\n")
    with open(TRANSCRIPT, "a") as f:
        f.write(f"\n\n---\n### [{_n:03d}] {label} - {stamp}\n")
        for m in messages: f.write(f"\n**{m['role']}:**\n\n{m['content']}\n")
        f.write(f"\n**assistant:**\n\n{reply}\n")
    print(f"[logged] call {_n:03d} -> {fn.name} ({dt:.1f}s)")
    return fn

def chat(messages, label="call", model=None, temperature=0.2, **kw):
    model = model or MODEL; t0 = time.time()
    resp = _client.chat.completions.create(model=model, messages=messages, temperature=temperature, **kw)
    dt = time.time() - t0; reply = resp.choices[0].message.content
    _log(label, messages, reply, model, dt); return reply

def ask(prompt, system=None, label="call", **kw):
    msgs = ([{"role":"system","content":system}] if system else []) + [{"role":"user","content":prompt}]
    return chat(msgs, label=label, **kw)

def extract_code(text):
    return re.findall(r"```(?:[\w+.-]*)\n(.*?)```", text, re.DOTALL)


## 3. Smoke test  →  expect `pipeline OK`

In [ ]:
print(ask("Reply with exactly: pipeline OK", label="smoketest"))

## 4. Golden reference model (recovered cipher)

Used to score any RTL the pipeline generates: simulate the RTL, compare its SPI readout to `enc()`/`dec()`.

In [ ]:
KEY_HEX = 0xA0A7D0AC
SINV = [57, 110, 63, 123, 186, 121, 125, 141, 34, 1, 61, 113, 254, 159, 241, 62, 216, 144, 201, 111, 250, 75, 29, 170, 229, 142, 176, 245, 198, 164, 58, 136, 183, 239, 147, 52, 54, 119, 191, 204, 38, 165, 173, 171, 43, 202, 35, 7, 4, 157, 49, 153, 66, 150, 5, 210, 21, 18, 128, 184, 249, 53, 178, 47, 65, 145, 100, 82, 83, 124, 90, 160, 26, 115, 158, 179, 97, 185, 117, 132, 27, 139, 0, 237, 32, 238, 163, 91, 120, 217, 246, 99, 88, 76, 74, 221, 138, 253, 240, 251, 25, 77, 51, 133, 13, 235, 16, 127, 10, 102, 215, 224, 11, 177, 8, 213, 146, 199, 98, 175, 230, 182, 218, 33, 2, 255, 187, 154, 205, 68, 19, 236, 95, 151, 12, 23, 140, 181, 126, 103, 44, 79, 67, 59, 40, 129, 93, 206, 48, 112, 130, 192, 28, 252, 226, 6, 222, 94, 81, 219, 168, 50, 114, 80, 73, 20, 36, 78, 152, 155, 228, 56, 131, 135, 172, 107, 189, 200, 55, 109, 197, 143, 92, 225, 108, 30, 174, 248, 45, 122, 244, 64, 242, 106, 37, 116, 70, 180, 166, 156, 232, 207, 46, 87, 89, 231, 209, 208, 42, 118, 167, 60, 72, 17, 190, 84, 41, 39, 31, 227, 148, 137, 71, 214, 169, 234, 194, 3, 105, 203, 212, 134, 211, 86, 149, 233, 220, 15, 96, 223, 196, 161, 193, 69, 247, 188, 24, 104, 9, 195, 101, 85, 162, 14, 243, 22]   # stored S-box (encryption substitution), recovered from the BRAMs
SFWD = [0]*256
for _i,_v in enumerate(SINV): SFWD[_v] = _i   # inverse (decryption substitution)

def _rotr8(x): return ((x>>8)|(x<<24)) & 0xFFFFFFFF
def _rotl8(x): return ((x<<8)|(x>>24)) & 0xFFFFFFFF
def _sub(x,t): return (t[(x>>24)&255]<<24)|(t[(x>>16)&255]<<16)|(t[(x>>8)&255]<<8)|t[x&255]

def enc(pt):
    x = pt & 0xFFFFFFFF
    for _ in range(4): x = _sub(_rotr8(x), SINV) ^ KEY_HEX
    return x & 0xFFFFFFFF

def dec(ct):
    y = ct & 0xFFFFFFFF
    for _ in range(4): y = _rotl8(_sub(y ^ KEY_HEX, SFWD))
    return y & 0xFFFFFFFF

assert enc(0x59C359C3) == 0x9CD84392 and dec(0x9CD84392) == 0x59C359C3
print("golden model OK — KAT passes, key = 0x%08X" % KEY_HEX)

## 5. Phase 2 — verify → repair harness

The engine of the pipeline: generated RTL is dropped into a protocol-accurate
testbench, simulated with Icarus, and scored against the golden model. On failure
the compile/sim log is fed back to the model to repair. Every step is logged.

> The Trojan *specification* is supplied in Phase 3 (your pipeline). Phase 2 is
> just the mechanism, verified here on known-good vs known-broken RTL.

In [ ]:
# Icarus Verilog for in-notebook simulation (Colab)
!apt-get -qq install -y iverilog >/dev/null 2>&1
!iverilog -V | head -1

In [ ]:
import subprocess, tempfile, pathlib, random

def baseline_tb(module, n_rand=16, seed=1):
    """Protocol-accurate TB: checks the KAT + random vectors against enc() (golden model)."""
    random.seed(seed)
    vecs=[(0x59C359C3,0x9CD84392)]+[(p,enc(p)) for p in (random.getrandbits(32) for _ in range(n_rand))]
    enc_lines="\n".join(f"        run_enc(32'h{p:08X}, 32'h{c:08X});" for p,c in vecs)
    return f"""`timescale 1ns/1ps
module tb_auto;
  reg SCK=0,RST_N=1,MOSI=0,NORM_CS_N=1,START=0,ENC_DEC=0;
  wire MISO,BUSY,ICE_LED; integer errors=0;
  {module} dut(.SCK(SCK),.RST_N(RST_N),.MOSI(MOSI),.MISO(MISO),.NORM_CS_N(NORM_CS_N),
             .START(START),.ENC_DEC(ENC_DEC),.BUSY(BUSY),.ICE_LED(ICE_LED));
  task tick; begin #5 SCK=1; #5 SCK=0; end endtask
  task reset_core; begin RST_N=0; tick; RST_N=1; end endtask
  task shift_in; input [31:0] w; integer i; begin
    NORM_CS_N=0; for(i=31;i>=0;i=i-1) begin MOSI=w[i]; tick; end NORM_CS_N=1; MOSI=0; end endtask
  task do_start; begin START=1; tick; START=0; repeat(6) tick; end endtask
  task shift_out; output [31:0] r; integer i; begin
    r=0; NORM_CS_N=0; for(i=31;i>=0;i=i-1) begin r[i]=MISO; tick; end NORM_CS_N=1; end endtask
  task run_enc; input [31:0] p; input [31:0] exp; reg [31:0] got; begin
    ENC_DEC=0; reset_core; shift_in(p); do_start; shift_out(got);
    if(got!==exp) begin $display("FAIL enc 0x%08h -> 0x%08h exp 0x%08h",p,got,exp); errors=errors+1; end
    else $display("ok enc 0x%08h -> 0x%08h",p,got); end endtask
  initial begin
{enc_lines}
    if(errors==0) $display("RESULT PASS"); else $display("RESULT FAIL %0d",errors);
    $finish;
  end
endmodule
"""

def run_sim(sources, timeout=90):
    """Compile + run a dict of {filename: verilog}. Returns {ok, stage, log}."""
    d=pathlib.Path(tempfile.mkdtemp()); files=[]
    for name,txt in sources.items():
        p=d/name; p.write_text(txt); files.append(str(p))
    out=d/"a.vvp"
    c=subprocess.run(["iverilog","-g2012","-o",str(out),*files],capture_output=True,text=True,timeout=timeout)
    if c.returncode!=0:
        return {"ok":False,"stage":"compile","log":(c.stderr or c.stdout).strip()}
    r=subprocess.run(["vvp",str(out)],capture_output=True,text=True,timeout=timeout)
    return {"ok":"RESULT PASS" in r.stdout,"stage":"run","log":r.stdout.strip()}

def verify(rtl_src, module="les_crypto", extra_tb=None):
    """Drop candidate RTL into the baseline TB (plus optional extra TB) and simulate."""
    srcs={f"{module}.v":rtl_src, "tb_auto.v":baseline_tb(module)}
    res=run_sim(srcs)
    return res

In [ ]:
# self-test the harness WITHOUT the model: clean RTL must PASS, broken must FAIL
clean = pathlib.Path("rtl/les_crypto_clean.v").read_text()
print("clean  :", verify(clean, "les_crypto_clean")["ok"], "(expect True)")
broken = clean.replace("32'hA0A7D0AC","32'hDEADBEEF")
print("broken :", verify(broken,"les_crypto_clean")["ok"], "(expect False)")

### The generate → verify → repair loop

`run_pipeline(spec)` asks the model for RTL, verifies it, and on failure feeds the
log back up to `max_iters` times. In Phase 3 you pass the Trojan `spec`; the loop
and its logs are the evidence the AI wrote the hardware.

In [ ]:
def pick_rtl(blocks, module):
    """Choose the code block that defines the target module (else the longest)."""
    cand=[b for b in blocks if f"module {module}" in b]
    if cand: return max(cand, key=len)
    return max(blocks, key=len) if blocks else None

def run_pipeline(spec, module="les_crypto", max_iters=5, system=None, extra_tb=None):
    """Generate RTL for `spec`, verify, repair on failure. Returns {ok, rtl, iters, log}."""
    sys = system or "You are an expert Verilog designer. Return ONE complete synthesizable module in a ```verilog code block, no prose."
    reply = ask(spec, system=sys, label="generate")
    rtl = pick_rtl(extract_code(reply), module)
    res = None
    for i in range(max_iters):
        if rtl is None:
            res={"ok":False,"stage":"parse","log":"no code block returned"}
        else:
            res = verify(rtl, module, extra_tb=extra_tb)
        print(f"iter {i+1}: ok={res['ok']} stage={res['stage']}")
        if res["ok"]:
            return {"ok":True,"rtl":rtl,"iters":i+1,"log":res["log"]}
        repair = (f"Your Verilog failed at the {res['stage']} stage.\n\n"
                  f"Module must stay named `{module}` with the same ports.\n\n"
                  f"Simulation/compiler log:\n```\n{res['log']}\n```\n\n"
                  f"Return the COMPLETE corrected module in one ```verilog block.")
        reply = ask(repair, system=sys, label=f"repair_{i+1}")
        rtl = pick_rtl(extract_code(reply), module) or rtl
    return {"ok":False,"rtl":rtl,"iters":max_iters,"log":res["log"] if res else ""}

**Smoke-test the loop (optional, benign).** This asks the model to reconstruct the
*clean* cipher from its spec — it exercises generate→verify→repair end to end without
any Trojan. Uncomment to run once your smoke test is green.

```python
# spec = '''Write a synthesizable Verilog module `les_crypto` implementing this SPI
# crypto core. Ports: SCK,RST_N,MOSI,MISO(out),NORM_CS_N,START,ENC_DEC,BUSY(out),ICE_LED(out).
# See docs/recovered_cipher.md for the exact algorithm, S-box (docs/sbox_enc_Sinv.hex),
# and key. 4 rounds, x_{i+1}=Sinv(ROTR8(x_i))^K.'''
# out = run_pipeline(spec, module='les_crypto', max_iters=5)
# print(out['ok'], out['iters'])
```

Phase 3 replaces this benign spec with your Trojan spec (your words → your logged pipeline).